In [ ]:
#| default_exp logger

# logger

> simple logger using idiomatic Solveit

In [ ]:
#| export
import json
from datetime import datetime
from html import unescape
from fastcore.all import patch
from dialoghelper.core import update_msg, find_msg_id, read_msg, find_dname
from dutil.core import set_var

In [ ]:
import random
import IPython.display
from IPython.display import Markdown
import fastcore.all as FC
from fastcore.test import *
from dutil.core import waitpred

In [ ]:
#| export
class Logger:
    "Timestamped logger that uses a cell's output as sink"
    msgid:str; dname:str
    def __init__(self, id:str='', dname:str='', clear:bool=True, sym='log'): 
        self.setup(id, dname, clear if not dname else False)
        self.show()
    def setup(self, id:str='', dname:str='', clear:bool=False, sym='log'): 
        "Setup logger for current message cell"
        curr = find_dname()
        self.dname, self.msgid = dname or curr, id or find_msg_id()
        self._xs = self.dname != curr
        if clear: self.clear()
        self._s = unescape(read_msg(0, id=self.msgid, dname=self.dname).output) if dname else getattr(self, '_s', '')
        set_var(sym, self, True)
    def show(self): self.msgid = find_msg_id(); print(self._s, end='')
    def clear(self): 
        "Clear all log entries and output"
        self._s=''; update_msg(self.msgid, output='', dname=self.dname)
    @property
    def logs(self): return self._s.splitlines()
    def __repr__(self): return self._s
    def __str__(self): return self._s
    def __call__(self, msg, *args, **kwargs): 
        "Add timestamped message to log"
        dt = datetime.now(); s = f"[{dt:%H:%M:%S}.{dt.microsecond//1000:03d}] {msg}"
        self._s = s + (f"\n{self._s}" if self._s != '' else '')
        out = '[{"name": "stdout", "output_type": "stream", "text": %s}]' % json.dumps(self._s)
        update_msg(self.msgid, output=out, dname=self.dname)

In [ ]:
if 'log' in globals(): del log
Logger()  # inject 'log'

[17:37:48.470] Some msg -> 610

In [ ]:
log('test')
log('test2')
log('test3')

In [ ]:
log(s := ';qwedcv fjkds')
test_is(s in log.logs[0], True)
log.logs

['[17:37:24.202] ;qwedcv fjkds',
 '[17:37:21.423] test3',
 '[17:37:21.397] test2',
 '[17:37:21.369] test']

In [ ]:
print(log, end='')  # __str__

[17:37:24.202] ;qwedcv fjkds
[17:37:21.423] test3
[17:37:21.397] test2
[17:37:21.369] test

In [ ]:
print(log._s, end='')

[17:37:24.202] ;qwedcv fjkds
[17:37:21.423] test3
[17:37:21.397] test2
[17:37:21.369] test

In [ ]:
log # __repr__

[17:37:24.202] ;qwedcv fjkds
[17:37:21.423] test3
[17:37:21.397] test2
[17:37:21.369] test

In [ ]:
log.clear()
test_eq(log.logs, [])

In [ ]:
log(f"Some msg -> {random.randint(0, 1000)}")
log.logs

['[17:37:48.470] Some msg -> 610']

In [ ]:
log.show()

In [ ]:
#| export
@patch
def show(self:Logger, clear:bool=False):
    "Display log in current cell, optionally clearing first"
    if self._xs: self._s = unescape(read_msg(0, id=self.msgid, dname=self.dname).output)
    else:
        if self.msgid != find_msg_id():
            oldid = self.msgid
            self.setup()
            update_msg(oldid, output='', dname=self.dname)
        if clear: self.clear()
    print(self._s, end='')

In [ ]:
log.show()

In [ ]:
log.show()

In [ ]:
for _ in "En algún lugar de la Mancha".split(): log(_)

In [ ]:
# across target
log.show()

[17:39:07.162] Mancha
[17:39:07.137] la
[17:39:07.111] de
[17:39:07.086] lugar
[17:39:07.060] algún
[17:39:07.034] En
[17:37:48.470] Some msg -> 610

Look at `show` method. What does it do?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The `show` method does three things:

1. **Checks if it's being called from a different cell than where the logger was last set up.** If so, it moves the logger to the new cell (updates `self.msgid` to the current cell) and clears the output from the old cell.

2. **Optionally clears the log** if `clear=True` is passed.

3. **Prints the current log content** (`self._s`) to the current cell's output.

This lets you call `log.show()` from any cell to display the accumulated logs there, and the logger will automatically "move" its display to that new cell while cleaning up the previous one.

Look at the `Logger` class and all the examples. What does it do?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The `Logger` class creates a simple logging system that displays timestamped messages in a cell's output area. Here's what it does:

**Core functionality:**
- Logs messages with timestamps (down to milliseconds)
- Stores messages in reverse chronological order (newest first)
- Updates a specific message cell's output in real-time as you add log entries

**Key features:**
- `log('message')` adds a timestamped entry
- `log.clear()` wipes all entries
- `log.show()` displays the log (and handles moving between cells)
- Messages are kept in `log.logs` as a list

**The clever bit:** It uses `update_msg()` to write directly to a cell's output, so you can log from anywhere in your notebook and see all the messages accumulate in one place. Notice how in the examples, multiple `log()` calls build up a running list with timestamps.

What aspect would you like to explore more? The timestamp formatting, how it tracks which cell to update, or something else?

# export -

In [ ]:
from dutil.flakes import show_flakes
show_flakes()

No warnings to report

In [ ]:
from dutil.gitutil import Git
g = Git('')

In [ ]:
g.cstatus()

On branch dev-log
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../dutil/_modidx.py
	modified:   ../dutil/core.py
	modified:   ../dutil/logger.py
	modified:   00_core.ipynb
	modified:   01_logger.ipynb

no changes added to commit (use "git add" and/or "git commit -a")


In [ ]:
g.cdiff('../dutil/logger.py')

diff --git a/dutil/logger.py b/dutil/logger.py
index cf5c635..294565d 100644
--- a/dutil/logger.py
+++ b/dutil/logger.py
@@ -11,33 +11,34 @@ from datetime import datetime
 from html import unescape
 from fastcore.all import patch
 from dialoghelper.core import update_msg, find_msg_id, read_msg, find_dname
-
+from .core import set_var
 
 # %% ../nbs/01_logger.ipynb #461b9dd1
 class Logger:
     "Timestamped logger that uses a cell's output as sink"
     msgid:str; dname:str
-    def __init__(self, id:str='', dname:str='', clear:bool=True): 
+    def __init__(self, id:str='', dname:str='', clear:bool=True, sym='log'): 
         self.setup(id, dname, clear if not dname else False)
-        print(self._s, end='')
-    def setup(self, id:str='', dname:str='', clear:bool=False): 
+        self.show()
+    def setup(self, id:str='', dname:str='', clear:bool=False, sym='log'): 
         "Setup logger for current message cell"
         curr = find_dname()
         self.dname, self.msgid = dname

In [ ]:
g.commit('-m', 'inject symbol "log" in user_ns; show on init')

In [ ]:
# #|hide
# #|eval: false
# from dutil.core import dlg_export
# dlg_export()